# 04 — Filter Bubble Simulation
Simulate 50-step user sessions with/without ε-greedy; plot ILD over time.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
from pathlib import Path
import faiss, json

faiss_path = Path('./data/faiss.index')
ids_path = Path('./data/faiss.index.ids.json')
if not faiss_path.exists():
    raise FileNotFoundError('Build FAISS index first')

index = faiss.read_index(str(faiss_path))
with open(ids_path) as f:
    paper_ids = json.load(f)

embeddings = {pid: np.array(index.reconstruct(i), dtype=np.float32) for i, pid in enumerate(paper_ids)}
print(f'Loaded {len(paper_ids)} paper embeddings')

In [ ]:
def cosine_dist(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return 1.0 - float(np.dot(a, b) / (na * nb)) if na > 0 and nb > 0 else 1.0

def ild(ids, embs):
    vs = [embs[p] for p in ids if p in embs]
    if len(vs) < 2: return 0.0
    return np.mean([cosine_dist(vs[i], vs[j]) for i in range(len(vs)) for j in range(i+1, len(vs))])

def simulate_session(use_exploration=False, n_steps=50, top_k=10, epsilon=0.1):
    """Greedy session: always click top-1, expand history, re-query."""
    seed_pid = random.choice(paper_ids)
    history = [seed_pid]
    ilds = []

    for _ in range(n_steps):
        user_emb = np.mean([embeddings[p] for p in history if p in embeddings], axis=0).astype(np.float32)
        norm = np.linalg.norm(user_emb)
        if norm > 0: user_emb /= norm
        D, I = index.search(user_emb.reshape(1, -1), top_k + len(history))
        recs = [paper_ids[i] for i in I[0] if i >= 0 and paper_ids[i] not in set(history)][:top_k]

        if use_exploration and random.random() < epsilon:
            inject = random.choice(paper_ids)
            if inject not in set(recs): recs[-1] = inject

        ilds.append(ild(recs, embeddings))
        history.append(recs[0] if recs else random.choice(paper_ids))

    return ilds

N_RUNS = 5
steps = range(1, 51)
base_ilds = np.mean([simulate_session(False) for _ in range(N_RUNS)], axis=0)
exp_ilds  = np.mean([simulate_session(True)  for _ in range(N_RUNS)], axis=0)

plt.figure(figsize=(10, 5))
plt.plot(steps, base_ilds, label='No exploration', color='tomato')
plt.plot(steps, exp_ilds,  label='ε-greedy (ε=0.1)', color='steelblue')
plt.xlabel('Session step'); plt.ylabel('ILD (avg pairwise cosine dist)')
plt.title('Filter Bubble Simulation: ILD over 50-step sessions')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('docs/filter_bubble_sim.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Final ILD — base: {base_ilds[-1]:.4f} | exploration: {exp_ilds[-1]:.4f}')